# 17 - OpenTripMap Candidate Generation

This notebook creates algorithmic candidate matches between:
- OSM-enriched tourism POIs
- OpenTripMap landmark results

Goal:
- narrow down plausible matches cheaply
- avoid full manual review of every OSM/OTM pair
- prepare a small candidate table for later review or LLM-assisted resolution


In [34]:
import ast
import math
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## Load sources

In [35]:
osm_df = pd.read_csv("../data/processed/poi_enriched.csv")
otm_df = pd.read_csv("../data/processed/opentripmap_radius_results.csv")

print("OSM:", osm_df.shape)
print("OTM:", otm_df.shape)


OSM: (2761, 20)
OTM: (35, 9)


## Keep only tourism-relevant OSM subset

In [36]:
osm_tourism = (
    osm_df[osm_df["category_clean"].isin(["museum", "historic", "attraction"])]
    .sort_values("importance_score", ascending=False)
    .copy()
)

osm_tourism[["poi_id", "name", "name_en", "category_clean", "importance_score"]].head(20)


,poi_id,name,name_en,category_clean,importance_score
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic,0.863004
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic,0.859224
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915
3,311681431,Sağlık Müzesi,NaN,museum,0.849310
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,museum,0.834391
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238
8,3373094254,Halı Müzesi,Carpet Museum,museum,0.827180
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic,0.821057


## Flatten OpenTripMap coordinates

In [37]:
point_parsed = otm_df["point"].apply(ast.literal_eval)
otm_df["lat"] = point_parsed.apply(lambda p: p.get("lat"))
otm_df["lon"] = point_parsed.apply(lambda p: p.get("lon"))

otm_df[["xid", "name", "lat", "lon", "query_area", "kinds"]].head(20)


,xid,name,lat,lon,query_area,kinds
0,N7215645385,The Blue Mosque,41.005253,28.976892,"Sultanahmet, Istanbul","religion,mosques,interesting_places"
1,W103953125,Tomb of Sultan Ahmet,41.006790,28.977037,"Sultanahmet, Istanbul","religion,mosques,interesting_places"
2,N415157636,Serpent Column,41.005661,28.975103,"Sultanahmet, Istanbul","historic,monuments_and_memorials,burial_places..."
3,N1768294290,Great Palace Mosaic Museum,41.004269,28.977449,"Sultanahmet, Istanbul","national_museums,cultural,museums,interesting_..."
4,N3236233297,Column of Marcian,41.015442,28.950285,"Eminonu, Istanbul","historic,monuments_and_memorials,burial_places..."
5,Q3082387,Church of the Virgin of the Pharos,41.005833,28.977222,"Sultanahmet, Istanbul","religion,churches,interesting_places,other_chu..."
6,N4988111427,Ruins of Bucoleon Palace,41.002533,28.976147,"Sultanahmet, Istanbul","palaces,architecture,historic_architecture,int..."
7,N4977947521,Church of St. Polyeuktos,41.014233,28.952959,"Eminonu, Istanbul","religion,architecture,historic_architecture,fo..."
8,Q475338,Great Palace of Constantinople,41.006390,28.977777,"Sultanahmet, Istanbul","palaces,architecture,historic_architecture,int..."
9,N3254460381,Aviation Martyrs' Monument,41.016075,28.952635,"Eminonu, Istanbul","historic,monuments_and_memorials,burial_places..."


## Text normalization helpers

In [38]:
GENERIC_WORDS = {
    "museum", "museums", "mosque", "mosques", "church", "churches",
    "palace", "palaces", "tower", "column", "cistern", "monument",
    "mausoleum", "tomb", "art", "history", "historic", "great",
    "the", "of", "and", "stone", "ruins",
    "müzesi", "muzesi", "camii", "cami", "sarayı", "sarayi", "saray",
    "kilisesi", "manastırı", "manastiri", "türbesi", "turbesi", "sarnıcı", "sarnici",
}


def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).casefold()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def meaningful_tokens(text):
    return {
        token
        for token in normalize_text(text).split()
        if token not in GENERIC_WORDS and len(token) >= 3
    }


def token_overlap_score(a, b):
    a_tokens = meaningful_tokens(a)
    b_tokens = meaningful_tokens(b)
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


## Distance helper

In [39]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))


## Light category compatibility

This is intentionally simple. It just helps remove obviously bad pairs.

In [40]:
def otm_bucket(kinds):
    kinds = str(kinds).casefold()
    if "museums" in kinds or "museum" in kinds:
        return "museum"
    if "monuments" in kinds or "monument" in kinds or "mausoleums" in kinds:
        return "historic"
    if "historic_architecture" in kinds or "palaces" in kinds or "architecture" in kinds:
        return "historic"
    if "mosques" in kinds or "churches" in kinds:
        return "attraction"
    return "other"


otm_df["otm_bucket"] = otm_df["kinds"].apply(otm_bucket)
otm_df[["name", "kinds", "otm_bucket"]].head(20)


,name,kinds,otm_bucket
0,The Blue Mosque,"religion,mosques,interesting_places",attraction
1,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",attraction
2,Serpent Column,"historic,monuments_and_memorials,burial_places...",historic
3,Great Palace Mosaic Museum,"national_museums,cultural,museums,interesting_...",museum
4,Column of Marcian,"historic,monuments_and_memorials,burial_places...",historic
5,Church of the Virgin of the Pharos,"religion,churches,interesting_places,other_chu...",attraction
6,Ruins of Bucoleon Palace,"palaces,architecture,historic_architecture,int...",historic
7,Church of St. Polyeuktos,"religion,architecture,historic_architecture,fo...",historic
8,Great Palace of Constantinople,"palaces,architecture,historic_architecture,int...",historic
9,Aviation Martyrs' Monument,"historic,monuments_and_memorials,burial_places...",historic


## Build candidate pairs

Rules:
- only compare tourism OSM rows to nearby OTM rows within a modest distance threshold
- require either category compatibility or some text overlap
- rank candidates by combined distance/text score


In [41]:
# Strict candidate set for higher-confidence matches
STRICT_MAX_DISTANCE_KM = 0.35
STRICT_MIN_TOKEN_OVERLAP = 0.10
STRICT_VERY_CLOSE_DISTANCE_KM = 0.05

# Broader review pool for human review
REVIEW_MAX_DISTANCE_KM = 0.60
REVIEW_MIN_TOKEN_OVERLAP = 0.00
REVIEW_VERY_CLOSE_DISTANCE_KM = 0.12
REVIEW_TARGET_ROWS = 200

strict_rows = []
review_rows = []

for _, osm_row in osm_tourism.iterrows():
    osm_name = osm_row["name"]
    osm_name_en = osm_row["name_en"] if pd.notna(osm_row["name_en"]) else ""

    for _, otm_row in otm_df.iterrows():
        dist_km = haversine_km(osm_row["lat"], osm_row["lon"], otm_row["lat"], otm_row["lon"])

        overlap_local = token_overlap_score(osm_name, otm_row["name"])
        overlap_en = token_overlap_score(osm_name_en, otm_row["name"])
        overlap = max(overlap_local, overlap_en)

        category_match = int(osm_row["category_clean"] == otm_row["otm_bucket"])

        base_record = {
            "poi_id": osm_row["poi_id"],
            "osm_name": osm_name,
            "osm_name_en": osm_name_en,
            "osm_category": osm_row["category_clean"],
            "osm_importance_score": osm_row["importance_score"],
            "otm_xid": otm_row["xid"],
            "otm_name": otm_row["name"],
            "otm_kinds": otm_row["kinds"],
            "otm_bucket": otm_row["otm_bucket"],
            "otm_query_area": otm_row["query_area"],
            "otm_wikidata": otm_row.get("wikidata"),
            "distance_km": dist_km,
            "token_overlap_score": overlap,
            "category_match": category_match,
        }

        # strict keep logic
        if dist_km <= STRICT_MAX_DISTANCE_KM:
            strict_keep = False
            if overlap >= STRICT_MIN_TOKEN_OVERLAP:
                strict_keep = True
            elif overlap > 0 and category_match == 1 and dist_km <= 0.15:
                strict_keep = True
            elif overlap == 0 and category_match == 1 and dist_km <= STRICT_VERY_CLOSE_DISTANCE_KM:
                strict_keep = True

            if strict_keep:
                distance_score = max(0.0, 1.0 - (dist_km / STRICT_MAX_DISTANCE_KM))
                combined_score = 0.60 * overlap + 0.30 * distance_score + 0.10 * category_match
                strict_rows.append(base_record | {
                    "distance_score": distance_score,
                    "combined_score": combined_score,
                })

        # broader review logic
        if dist_km <= REVIEW_MAX_DISTANCE_KM:
            review_keep = False
            if overlap >= 0.20:
                review_keep = True
            elif overlap >= 0.10 and dist_km <= 0.35:
                review_keep = True
            elif overlap > 0 and category_match == 1 and dist_km <= 0.25:
                review_keep = True
            elif category_match == 1 and dist_km <= REVIEW_VERY_CLOSE_DISTANCE_KM:
                review_keep = True

            if review_keep:
                distance_score = max(0.0, 1.0 - (dist_km / REVIEW_MAX_DISTANCE_KM))
                review_score = 0.50 * overlap + 0.25 * distance_score + 0.15 * category_match + 0.10 * float(osm_row["importance_score"])
                review_rows.append(base_record | {
                    "distance_score": distance_score,
                    "combined_score": review_score,
                })

strict_candidates_df = pd.DataFrame(strict_rows)
review_candidates_df = pd.DataFrame(review_rows)
print("Strict candidate rows:", strict_candidates_df.shape)
print("Review candidate rows:", review_candidates_df.shape)
review_candidates_df.head(20)


Strict candidate rows: (32, 16)
Review candidate rows: (61, 16)


,poi_id,osm_name,osm_name_en,osm_category,osm_importance_score,otm_xid,otm_name,otm_kinds,otm_bucket,otm_query_area,otm_wikidata,distance_km,token_overlap_score,category_match,distance_score,combined_score
0,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915,N415157636,Serpent Column,"historic,monuments_and_memorials,burial_places...",historic,"Sultanahmet, Istanbul",Q588892,0.085547,0.000000,1,0.857422,0.450247
1,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915,N415157637,Obelisk of Theodosius,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q763854,0.072467,0.000000,1,0.879222,0.455697
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915,N415156590,The Walled Obelisk,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q742474,0.111155,0.000000,1,0.814742,0.439577
3,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915,Q1152384,Palace of Daphne,"palaces,architecture,historic_architecture,int...",historic,"Sultanahmet, Istanbul",Q1152384,0.109298,0.000000,1,0.817837,0.440351
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213,N1768294290,Great Palace Mosaic Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q2719230,0.003208,1.000000,1,0.994653,0.983085
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898,W103953125,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.004367,1.000000,1,0.992722,0.981770
6,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898,Q3082387,Church of the Virgin of the Pharos,"religion,churches,interesting_places,other_chu...",attraction,"Sultanahmet, Istanbul",Q3082387,0.108227,0.000000,1,0.819622,0.438495
7,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,museum,0.834391,R8120955,Turk and Islamic Art Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.038256,0.000000,1,0.936240,0.467499
8,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,museum,0.834391,N5113500256,Museum of Turkish and Islamic arts,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.064798,0.000000,1,0.892003,0.456440
9,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238,R8120955,Turk and Islamic Art Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.015997,0.250000,1,0.973339,0.601759


## Keep top candidates per OSM POI

In [42]:
if not strict_candidates_df.empty:
    top_candidates_df = (
        strict_candidates_df.sort_values(
            ["poi_id", "combined_score", "token_overlap_score", "distance_km"],
            ascending=[True, False, False, True],
        )
        .groupby("poi_id", as_index=False)
        .head(4)
        .reset_index(drop=True)
    )
else:
    top_candidates_df = strict_candidates_df.copy()

if not review_candidates_df.empty:
    review_pool_df = (
        review_candidates_df.sort_values(
            ["combined_score", "token_overlap_score", "distance_km"],
            ascending=[False, False, True],
        )
        .head(REVIEW_TARGET_ROWS)
        .reset_index(drop=True)
    )
else:
    review_pool_df = review_candidates_df.copy()

print("Top strict candidate rows:", top_candidates_df.shape)
print("Review pool rows:", review_pool_df.shape)
review_pool_df.head(30)


Top strict candidate rows: (32, 16)
Review pool rows: (61, 16)


,poi_id,osm_name,osm_name_en,osm_category,osm_importance_score,otm_xid,otm_name,otm_kinds,otm_bucket,otm_query_area,otm_wikidata,distance_km,token_overlap_score,category_match,distance_score,combined_score
0,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213,N1768294290,Great Palace Mosaic Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q2719230,0.003208,1.000000,1,0.994653,0.983085
1,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898,W103953125,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.004367,1.000000,1,0.992722,0.981770
2,1908677124,Milyon Taşı,Milion,historic,0.796279,N1908677124,Milion,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.002926,1.000000,1,0.995123,0.978409
3,1120852417,Theodosius Dikilitaşı,Obelisk of Theodosius,historic,0.787757,N415157637,Obelisk of Theodosius,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q763854,0.002675,1.000000,1,0.995542,0.977661
4,1120852418,Örme Dikilitaş,The Walled Obelisk,historic,0.772782,N415156590,The Walled Obelisk,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q742474,0.002100,1.000000,1,0.996501,0.976403
5,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238,N5113500256,Museum of Turkish and Islamic arts,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.022442,1.000000,1,0.962597,0.974073
6,1908677124,Milyon Taşı,Milion,historic,0.796279,N5108843107,Milion Stone,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.015177,1.000000,1,0.974705,0.973304
7,18055570,Sultanahmet Camii,Blue Mosque,attraction,0.807981,N7215645385,The Blue Mosque,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.026609,1.000000,1,0.955652,0.969711
8,1184851395,Bukoleon Sarayı Kalıntıları,Bucoleon Palace Ruins,historic,0.736480,N4988111427,Ruins of Bucoleon Palace,"palaces,architecture,historic_architecture,int...",historic,"Sultanahmet, Istanbul",Q1003322,0.009958,1.000000,1,0.983403,0.969499
9,4977947521,Ayios Polieuktos Kilisesi,Church of St. Polyeuktos,historic,0.641801,N4977947521,Church of St. Polyeuktos,"religion,architecture,historic_architecture,fo...",historic,"Eminonu, Istanbul",Q1578614,0.000082,1.000000,1,0.999864,0.964146


## Quick inspection of strongest candidates

In [43]:
print('Strict shortlist preview:')
display(top_candidates_df.sort_values(["combined_score", "token_overlap_score"], ascending=[False, False]).head(40))

print('Review pool preview:')
display(review_pool_df.sort_values(["combined_score", "token_overlap_score"], ascending=[False, False]).head(60))


Strict shortlist preview:


,poi_id,osm_name,osm_name_en,osm_category,osm_importance_score,otm_xid,otm_name,otm_kinds,otm_bucket,otm_query_area,otm_wikidata,distance_km,token_overlap_score,category_match,distance_score,combined_score
27,4977947521,Ayios Polieuktos Kilisesi,Church of St. Polyeuktos,historic,0.641801,N4977947521,Church of St. Polyeuktos,"religion,architecture,historic_architecture,fo...",historic,"Eminonu, Istanbul",Q1578614,0.000082,1.000000,1,0.999766,0.999930
25,3236233297,Kıztaşı,Column of Marcian,historic,0.614550,N3236233297,Column of Marcian,"historic,monuments_and_memorials,burial_places...",historic,"Eminonu, Istanbul",Q285236,0.000093,1.000000,1,0.999733,0.999920
17,1120852418,Örme Dikilitaş,The Walled Obelisk,historic,0.772782,N415156590,The Walled Obelisk,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q742474,0.002100,1.000000,1,0.994001,0.998200
14,1120852417,Theodosius Dikilitaşı,Obelisk of Theodosius,historic,0.787757,N415157637,Obelisk of Theodosius,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q763854,0.002675,1.000000,1,0.992358,0.997707
23,1908677124,Milyon Taşı,Milion,historic,0.796279,N1908677124,Milion,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.002926,1.000000,1,0.991640,0.997492
20,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213,N1768294290,Great Palace Mosaic Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q2719230,0.003208,1.000000,1,0.990835,0.997250
3,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898,W103953125,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.004367,1.000000,1,0.987524,0.996257
21,1184851395,Bukoleon Sarayı Kalıntıları,Bucoleon Palace Ruins,historic,0.736480,N4988111427,Ruins of Bucoleon Palace,"palaces,architecture,historic_architecture,int...",historic,"Sultanahmet, Istanbul",Q1003322,0.009958,1.000000,1,0.971548,0.991465
24,1908677124,Milyon Taşı,Milion,historic,0.796279,N5108843107,Milion Stone,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.015177,1.000000,1,0.956637,0.986991
29,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238,N5113500256,Museum of Turkish and Islamic arts,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.022442,1.000000,1,0.935880,0.980764


Review pool preview:


,poi_id,osm_name,osm_name_en,osm_category,osm_importance_score,otm_xid,otm_name,otm_kinds,otm_bucket,otm_query_area,otm_wikidata,distance_km,token_overlap_score,category_match,distance_score,combined_score
0,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213,N1768294290,Great Palace Mosaic Museum,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q2719230,0.003208,1.000000,1,0.994653,0.983085
1,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898,W103953125,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.004367,1.000000,1,0.992722,0.981770
2,1908677124,Milyon Taşı,Milion,historic,0.796279,N1908677124,Milion,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.002926,1.000000,1,0.995123,0.978409
3,1120852417,Theodosius Dikilitaşı,Obelisk of Theodosius,historic,0.787757,N415157637,Obelisk of Theodosius,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q763854,0.002675,1.000000,1,0.995542,0.977661
4,1120852418,Örme Dikilitaş,The Walled Obelisk,historic,0.772782,N415156590,The Walled Obelisk,"historic,monuments_and_memorials,urban_environ...",historic,"Sultanahmet, Istanbul",Q742474,0.002100,1.000000,1,0.996501,0.976403
5,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238,N5113500256,Museum of Turkish and Islamic arts,"national_museums,cultural,museums,interesting_...",museum,"Sultanahmet, Istanbul",Q525939,0.022442,1.000000,1,0.962597,0.974073
6,1908677124,Milyon Taşı,Milion,historic,0.796279,N5108843107,Milion Stone,"milestones,architecture,historic_architecture,...",historic,"Sultanahmet, Istanbul",Q1187329,0.015177,1.000000,1,0.974705,0.973304
7,18055570,Sultanahmet Camii,Blue Mosque,attraction,0.807981,N7215645385,The Blue Mosque,"religion,mosques,interesting_places",attraction,"Sultanahmet, Istanbul",Q80541,0.026609,1.000000,1,0.955652,0.969711
8,1184851395,Bukoleon Sarayı Kalıntıları,Bucoleon Palace Ruins,historic,0.736480,N4988111427,Ruins of Bucoleon Palace,"palaces,architecture,historic_architecture,int...",historic,"Sultanahmet, Istanbul",Q1003322,0.009958,1.000000,1,0.983403,0.969499
9,4977947521,Ayios Polieuktos Kilisesi,Church of St. Polyeuktos,historic,0.641801,N4977947521,Church of St. Polyeuktos,"religion,architecture,historic_architecture,fo...",historic,"Eminonu, Istanbul",Q1578614,0.000082,1.000000,1,0.999864,0.964146


## Save review file

In [44]:
if top_candidates_df.empty:
    print("Skipping save because no strict candidates were generated.")
else:
    top_candidates_df.to_csv("../data/processed/otm_osm_candidate_matches_strict.csv", index=False)
    print("Saved: ../data/processed/otm_osm_candidate_matches_strict.csv")
    print("Shape:", top_candidates_df.shape)

if review_pool_df.empty:
    print("Skipping review-pool save because no review candidates were generated.")
else:
    review_pool_df.to_csv("../data/processed/otm_osm_candidate_matches_review_pool.csv", index=False)
    print("Saved: ../data/processed/otm_osm_candidate_matches_review_pool.csv")
    print("Shape:", review_pool_df.shape)


Saved: ../data/processed/otm_osm_candidate_matches_strict.csv
Shape: (32, 16)
Saved: ../data/processed/otm_osm_candidate_matches_review_pool.csv
Shape: (61, 16)
